# SGCarMart Used Cars Scraper

Scrapes all used car listings from [sgcarmart.com/used-cars](https://www.sgcarmart.com/used-cars/) using **TinyFish AI** (goal-based browser agent with stealth mode + residential proxies).

**Prerequisites:**
- `TINYFISH_API_KEY` environment variable set
- `pip install httpx pandas pyarrow`

**Strategy:**
1. Explore site structure to confirm pagination URL pattern
2. Scrape each listing page (100 cars/page) via TinyFish
3. Clean and deduplicate
4. Export to CSV + Parquet

In [1]:
%pip install -q httpx pandas pyarrow

Note: you may need to restart the kernel to use updated packages.


In [6]:
import os
import json
import time
import httpx
import pandas as pd
from typing import Optional
from dotenv import load_dotenv

load_dotenv()

# ── Config ────────────────────────────────────────────────────────────────────
TINYFISH_API_KEY  = os.environ.get("TINYFISH_API_KEY", "")
TINYFISH_ENDPOINT = "https://agent.tinyfish.ai/v1/automation/run-sse"

SGCARMART_BASE    = "https://www.sgcarmart.com/used-cars/"
LISTINGS_PER_PAGE = 100   # SGCarMart supports up to 100
REQUEST_DELAY     = 3.0   # seconds between TinyFish calls

assert TINYFISH_API_KEY, (
    "Set TINYFISH_API_KEY environment variable before running.\n"
    "  export TINYFISH_API_KEY=tf-xxxx"
)
print("Config OK. API key present.")

Config OK. API key present.


## TinyFish SSE Client

Streams `POST /v1/automation/run-sse` and returns the `result` from the final `COMPLETE` event.

In [7]:
def run_tinyfish(
    url: str,
    goal: str,
    browser_profile: str = "stealth",
    timeout: int = 300,
    verbose: bool = True,
) -> dict | list | None:
    """
    Run a TinyFish browser automation task via SSE streaming.

    Returns the parsed result object from the COMPLETE event.
    Raises RuntimeError if the task fails or is cancelled.
    """
    headers = {
        "X-API-Key": TINYFISH_API_KEY,
        "Content-Type": "application/json",
        "Accept": "text/event-stream",
    }
    payload = {
        "url": url,
        "goal": goal,
        "browser_profile": browser_profile,
    }

    with httpx.Client(timeout=timeout) as client:
        with client.stream("POST", TINYFISH_ENDPOINT, json=payload, headers=headers) as resp:
            resp.raise_for_status()
            for line in resp.iter_lines():
                if not line or line.startswith(":") or not line.startswith("data: "):
                    continue
                try:
                    event = json.loads(line[6:])
                except json.JSONDecodeError:
                    continue

                etype = event.get("type", "")
                if etype == "STARTED" and verbose:
                    print(f"[TinyFish] run_id={event.get('run_id')}")
                elif etype == "PROGRESS" and verbose:
                    print(f"  → {event.get('purpose', '')}")
                elif etype == "COMPLETE":
                    status = event.get("status")
                    if status == "COMPLETED":
                        raw = event.get("result")
                        if isinstance(raw, str):
                            try:
                                return json.loads(raw)
                            except json.JSONDecodeError:
                                return raw
                        return raw
                    else:
                        err = event.get("error") or event.get("help_message") or "Unknown"
                        raise RuntimeError(f"TinyFish task {status}: {err}")
    return None

## Step 1 — Explore Site Structure

One call to confirm pagination URL pattern and data fields available per listing.

In [10]:
print("Exploring SGCarMart listing structure...")

explore_result = run_tinyfish(
    url=SGCARMART_BASE,
    goal="""
Visit this used car listing page.
1. What data fields are shown per car listing (e.g. brand, model, year, price, mileage)?
2. How many car listings are shown on this page?
3. What is the URL pattern used for pagination? Look at 'Next' page links or numbered page links.
4. What is the total number of listings shown anywhere on the page?

Return ONLY a JSON object:
{
  "data_fields": [list of field names visible per car listing],
  "listings_on_this_page": <integer>,
  "pagination_url_pattern": "example URL for page 2",
  "total_listings": <integer or null>
}
""",
)
print(json.dumps(explore_result, indent=2))

Exploring SGCarMart listing structure...
[TinyFish] run_id=0a2a59bd-77e4-403d-8bd5-95f6a15a9c61


KeyboardInterrupt: 

## Step 2 — Scraper Functions

In [11]:
# SGCarMart pagination: ?BRSR=<offset>&RPG=<per_page>&AVG=0
# BRSR = Begin Row (0-indexed), RPG = Results Per Group

def build_listing_url(offset: int, per_page: int = LISTINGS_PER_PAGE) -> str:
    return f"{SGCARMART_BASE}?BRSR={offset}&RPG={per_page}&AVG=0"


EXTRACTION_GOAL = """
This is a used car listings page on SGCarMart. Extract ALL car listings visible.

For EVERY listing, extract these fields (null if not shown):
- listing_url:       full URL to the car detail page (the href of the listing title/link)
- title:             full listing headline text
- brand:             car make (e.g. Toyota, BMW, Honda)
- model:             car model (e.g. Corolla, 3 Series)
- variant:           trim/variant if shown
- year:              registration or manufactured year (integer)
- price:             asking price in SGD (integer, digits only, no commas/symbols)
- mileage_km:        odometer in km (integer, digits only)
- transmission:      Auto or Manual
- fuel_type:         Petrol / Diesel / Electric / Hybrid / Petrol-Electric
- engine_cc:         engine displacement in cc (integer)
- vehicle_type:      body type (Sedan, SUV, Hatchback, MPV, Coupe, etc.)
- num_owners:        number of previous owners (integer)
- coe_expiry:        COE expiry date string if shown
- deregister_value:  PARF/deregistration value in SGD (integer) if shown
- dealer_name:       seller or dealership name
- is_direct_owner:   true if sold directly by owner, false if by dealer
- listing_date:      date listing was posted (string)

Do NOT skip or truncate listings. Extract every single one on the page.

Return ONLY a JSON object:
{"listings": [{...}, ...], "count": <number of listings extracted>}
"""


def scrape_page(offset: int) -> list[dict]:
    """Scrape one listing page at the given row offset."""
    url = build_listing_url(offset)
    print(f"  URL: {url}")

    result = run_tinyfish(url=url, goal=EXTRACTION_GOAL, browser_profile="stealth")

    if result is None:
        print("  Warning: null result")
        return []

    if isinstance(result, list):
        listings = result
    elif isinstance(result, dict):
        listings = result.get("listings") or result.get("data") or []
    else:
        print(f"  Warning: unexpected result type {type(result)}")
        return []

    for item in listings:
        item["_page_offset"] = offset

    print(f"  Extracted {len(listings)} listings")
    return listings

In [12]:
def scrape_all(
    max_pages: Optional[int] = None,
    per_page: int = LISTINGS_PER_PAGE,
    delay: float = REQUEST_DELAY,
) -> pd.DataFrame:
    """
    Paginate through all SGCarMart used car listings.

    Args:
        max_pages: Stop after N pages (None = all pages).
        per_page:  Listings per page (max 100).
        delay:     Seconds between TinyFish API calls.
    """
    all_listings: list[dict] = []
    page = 0

    while True:
        offset = page * per_page
        print(f"\n── Page {page + 1} (offset {offset}) {'─' * 40}")

        try:
            listings = scrape_page(offset)
        except RuntimeError as e:
            print(f"  TinyFish error: {e}")
            break
        except Exception as e:
            print(f"  Unexpected error: {e}")
            break

        if not listings:
            print("  No listings returned — end of results.")
            break

        all_listings.extend(listings)
        print(f"  Running total: {len(all_listings)}")

        if len(listings) < per_page:
            print("  Partial page — reached last page.")
            break

        page += 1
        if max_pages is not None and page >= max_pages:
            print(f"  Reached max_pages={max_pages} limit.")
            break

        time.sleep(delay)

    print(f"\nDone. Total: {len(all_listings)} listings")
    return pd.DataFrame(all_listings)

## Step 3 — Run Scraper

`max_pages=2` scrapes ~200 listings to validate. Remove the limit for a full run.

In [13]:
# Validation: first 2 pages (~200 listings)
df_raw = scrape_all(max_pages=2)

print(f"\nShape   : {df_raw.shape}")
print(f"Columns : {list(df_raw.columns)}")
df_raw.head(3)


── Page 1 (offset 0) ────────────────────────────────────────
  URL: https://www.sgcarmart.com/used-cars/?BRSR=0&RPG=100&AVG=0
[TinyFish] run_id=d1f9d7a4-f63e-4dac-9abd-9c3d4ee70d44
  → Visit SGCarMart used cars listing page.
  → Extract car listings and their details from the current page.
  → Extract all 100 car listings with all required fields.
  → Check the layout of the page to identify car listings.
  → Extract all car listings from all sections on the page.
  → Extract all unique car listings from the page.
  → Extract 'Latest Cars Added' section.
  Extracted 71 listings
  Running total: 71
  Partial page — reached last page.

Done. Total: 71 listings

Shape   : (71, 19)
Columns : ['listing_url', 'title', 'brand', 'model', 'variant', 'year', 'price', 'mileage_km', 'transmission', 'fuel_type', 'engine_cc', 'vehicle_type', 'num_owners', 'coe_expiry', 'deregister_value', 'dealer_name', 'is_direct_owner', 'listing_date', '_page_offset']


,listing_url,title,brand,model,variant,year,price,mileage_km,transmission,fuel_type,engine_cc,vehicle_type,num_owners,coe_expiry,deregister_value,dealer_name,is_direct_owner,listing_date,_page_offset
0,https://www.sgcarmart.com/used-cars/info/bmw-1...,BMW 1 Series 118i 5DR Highline,BMW,1 Series,118i 5DR Highline,None,85000.0,None,None,None,NaN,None,NaN,None,None,None,False,None,0
1,https://www.sgcarmart.com/used-cars/info/mazda...,Mazda Biante 2.0A,Mazda,Biante,2.0A,None,17888.0,None,None,None,NaN,None,NaN,None,None,None,False,None,0
2,https://www.sgcarmart.com/used-cars/info/toyot...,Toyota Harrier 2.0A Elegance Panoramic Roof (N...,Toyota,Harrier,2.0A Elegance Panoramic Roof,None,138882.0,None,None,None,NaN,None,NaN,None,None,None,False,None,0


In [ ]:
# Full run — uncomment once validation passes
# df_raw = scrape_all()

In [17]:
df_raw.head(50)

,listing_url,title,brand,model,variant,year,price,mileage_km,transmission,fuel_type,engine_cc,vehicle_type,num_owners,coe_expiry,deregister_value,dealer_name,is_direct_owner,listing_date,_page_offset
0,https://www.sgcarmart.com/used-cars/info/bmw-1...,BMW 1 Series 118i 5DR Highline,BMW,1 Series,118i 5DR Highline,None,85000.0,None,None,None,NaN,None,NaN,None,None,None,False,None,0
1,https://www.sgcarmart.com/used-cars/info/mazda...,Mazda Biante 2.0A,Mazda,Biante,2.0A,None,17888.0,None,None,None,NaN,None,NaN,None,None,None,False,None,0
2,https://www.sgcarmart.com/used-cars/info/toyot...,Toyota Harrier 2.0A Elegance Panoramic Roof (N...,Toyota,Harrier,2.0A Elegance Panoramic Roof,None,138882.0,None,None,None,NaN,None,NaN,None,None,None,False,None,0
3,https://www.sgcarmart.com/used-cars/info/merce...,Mercedes-Benz E-Class E250 Exclusive,Mercedes-Benz,E-Class,E250 Exclusive,None,99000.0,None,None,None,NaN,None,NaN,None,None,None,False,None,0
4,https://www.sgcarmart.com/used-cars/info/volks...,Volkswagen Sportsvan 1.4A TSI Comfortline,Volkswagen,Sportsvan,1.4A TSI Comfortline,None,16000.0,None,None,None,NaN,None,NaN,None,None,None,False,None,0
5,https://www.sgcarmart.com/used-cars/info/audi-...,Audi A6 Mild Hybrid 2.0A TFSI S-tronic Design,Audi,A6,Mild Hybrid 2.0A TFSI S-tronic Design,None,198888.0,None,None,Hybrid,2000.0,None,NaN,None,None,None,False,None,0
6,https://www.sgcarmart.com/used-cars/info/toyot...,Toyota Camry 2.5A (New 5-yr COE),Toyota,Camry,2.5A,None,89800.0,None,None,None,NaN,None,NaN,None,None,None,False,None,0
7,https://www.sgcarmart.com/used-cars/info/honda...,Honda Civic 1.5A VTEC Turbo Sunroof,Honda,Civic,1.5A VTEC Turbo Sunroof,None,80800.0,None,None,Petrol,1500.0,None,NaN,None,None,None,False,None,0
8,https://www.sgcarmart.com/used-cars/info/merce...,Mercedes-Benz C-Class C200K (COE till 05/2028),Mercedes-Benz,C-Class,C200K,None,22800.0,None,None,None,NaN,None,NaN,05/2028,None,None,False,None,0
9,https://www.sgcarmart.com/used-cars/info/bmw-2...,BMW 2 Series 216i Gran Tourer,BMW,2 Series,216i Gran Tourer,None,54800.0,None,None,None,NaN,None,NaN,None,None,None,False,None,0


## Step 4 — Clean & Normalize

In [ ]:
def clean_listings(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize types, strip whitespace, fix URLs, deduplicate."""
    df = df.copy()

    # Numeric
    for col in ["price", "mileage_km", "engine_cc", "year", "num_owners", "deregister_value"]:
        if col in df.columns:
            df[col] = pd.to_numeric(
                df[col].astype(str).str.replace(r"[^\d.]", "", regex=True).replace("", None),
                errors="coerce"
            )
            if col in ("year", "num_owners"):
                df[col] = df[col].astype("Int64")

    # Strings
    for col in ["brand", "model", "variant", "transmission", "fuel_type", "vehicle_type", "dealer_name"]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().str.title().replace({"None": None, "Nan": None})

    # Boolean
    if "is_direct_owner" in df.columns:
        df["is_direct_owner"] = df["is_direct_owner"].map(
            lambda x: True  if str(x).lower() in ("true", "1", "yes") else
                      False if str(x).lower() in ("false", "0", "no")  else None
        )

    # Fix relative listing URLs
    if "listing_url" in df.columns:
        def fix_url(u):
            if pd.isna(u) or not str(u).strip():
                return None
            u = str(u).strip()
            return u if u.startswith("http") else "https://www.sgcarmart.com" + (u if u.startswith("/") else "/" + u)
        df["listing_url"] = df["listing_url"].apply(fix_url)

    # Deduplicate
    before = len(df)
    key = "listing_url" if "listing_url" in df.columns else None
    df = df.drop_duplicates(subset=[key]) if key else df.drop_duplicates()
    print(f"Deduplicated: {before} → {len(df)} rows")

    # Column order
    priority = ["brand", "model", "variant", "year", "price", "mileage_km",
                "transmission", "fuel_type", "engine_cc", "vehicle_type",
                "num_owners", "coe_expiry", "deregister_value",
                "dealer_name", "is_direct_owner", "listing_date", "listing_url", "title"]
    ordered = [c for c in priority if c in df.columns]
    df = df[ordered + [c for c in df.columns if c not in ordered]]

    if "price" in df.columns:
        df = df.sort_values("price", na_position="last").reset_index(drop=True)

    return df


df = clean_listings(df_raw)
print(f"Final shape: {df.shape}")
df.dtypes

In [ ]:
df.describe(include="all")

## Step 5 — Export

In [ ]:
import pathlib

out = pathlib.Path("output")
out.mkdir(exist_ok=True)

csv_path     = out / "sgcarmart_used_cars.csv"
parquet_path = out / "sgcarmart_used_cars.parquet"

df.to_csv(csv_path, index=False)
df.to_parquet(parquet_path, index=False)

print(f"Saved:")
print(f"  CSV     → {csv_path}  ({csv_path.stat().st_size / 1024:.1f} KB)")
print(f"  Parquet → {parquet_path}  ({parquet_path.stat().st_size / 1024:.1f} KB)")

print(f"\n── Summary {'─' * 40}")
print(f"Total listings : {len(df):,}")
if "brand" in df.columns:
    print(f"\nTop 10 brands:\n{df['brand'].value_counts().head(10).to_string()}")
if "price" in df.columns:
    p = df['price'].dropna()
    print(f"\nPrice (SGD)  min={p.min():,.0f}  median={p.median():,.0f}  max={p.max():,.0f}")
if "mileage_km" in df.columns:
    m = df['mileage_km'].dropna()
    print(f"Mileage (km) min={m.min():,.0f}  median={m.median():,.0f}  max={m.max():,.0f}")